# PSX Financial Distress Early Warning System
## Four-Stage Distress Classification: SMOTE vs Hierarchical Decomposition
### Pakistan Stock Exchange (PSX) | FY2011–FY2024 | 349 Non-Financial Firms

---

**Data Source**: Financial Statements Analysis of the Non-Financial Companies (FSANFC), State Bank of Pakistan (2023, 2024, 2025).

**Reproducibility**: All stochastic components are seeded with `random_state=42`. Run cells sequentially from §1 to §16. Google Drive mount (§2) is required for checkpoint and output persistence.

---

### Pipeline Overview

| Section | Description |
|---------|-------------|
| §1 | Install dependencies and imports |
| §2 | Mount Google Drive and set working directory |
| §3 | Load and validate raw FSANFC data |
| §4 | Target engineering: create T+1 forward-looking distress labels |
| §5 | Cleaning pipeline: missingness filter, correlation filter (ρ ≥ 0.98) |
| §6 | Temporal split: last 20% per class reserved for test set |
| §7 | Custom transformers: firm-wise rolling mean imputation + RobustScaler |
| §8 | Mutual information feature selection: Top-6 per binary flag (1-vs-Rest) |
| §8b | Layer-specific feature sets for hierarchical decomposition |
| §8c | Feature importance visualisation |
| §9 | Model configurations: six tree-based and ensemble classifiers |
| §10 | Evaluation framework: Macro Recall, per-stage recall, train–test gap |
| §11 | **Experiment A** — SMOTE: flat four-class classification |
| §12 | **Experiment B** — Hierarchical Decomposition Strategy 1 |
| §13 | Master comparison table and best model selection |
| §14 | Detailed visualisations: confusion matrices, performance charts |
| §15-XAI | Twelve-module Explainable AI (XAI) audit |
| §16 | Save all artefacts to Google Drive |

---

### Key Design Decisions

**Four-stage taxonomy** extends Farooq, Qamar & Haque (2018) by adding an explicit Healthy class (Stage 0), enabling the model to distinguish Profit Reduction (Stage 1), Mild Liquidity Distress (Stage 2), and Severe Liquidity Distress (Stage 3).

**SMOTE applied within the cross-validation pipeline** (not to the full dataset) to prevent synthetic data leakage into validation folds (Chawla et al., 2002).

**Hierarchical decomposition (Strategy 1)** groups by financial severity: Layer 1 separates {Stage 0, Stage 1} from {Stage 2, Stage 3}; Layer 2a distinguishes Stage 0 vs Stage 1; Layer 2b distinguishes Stage 2 vs Stage 3. Sub-models are trained on Layer 1 predicted routing to correct exposure bias.

**Temporal split** preserves chronological order: the last 20% of observations within each stage are held out for testing, ensuring no future information leaks into training.

**Firm-wise rolling mean imputation** uses only past observations within each firm to respect the time-series structure and prevent future leakage.


In [ ]:
# §1 — Install Dependencies & Imports
# Reason: pinning minimal set of libraries; imbalanced-learn for SMOTE,
#         lightgbm/xgboost for gradient boosting, shap for explainability.
!pip install xgboost lightgbm imbalanced-learn shap scikit-learn openpyxl -q

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ── Sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, StratifiedShuffleSplit
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    balanced_accuracy_score, accuracy_score,
    ConfusionMatrixDisplay, roc_auc_score,
    classification_report
)
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    BaggingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

try:
    import shap; SHAP_OK = True
except ImportError:
    SHAP_OK = False; print('shap not installed')

# ── Global constants
FIRM_COL    = 'Organization Name'
YEAR_COL    = 'Fiscal_Year'
MAIN_TARGET = 'Main_Distress'          # current-year 4-class (used as feature too)
TARGET_COL  = 'Main_Distress_T1'       # prediction target (T+1 shift)

BINARY_COLS = [
    'Profit_Reduction',
    'Mild_Liquidity_Distress',
    'Severe_Liquidity_Distress',
]
T1_COLS  = [f'{c}_T1' for c in BINARY_COLS]
# Healthy is the 0 class of Main_Distress — derived later, not a separate column

STAGE_LABELS = [0, 1, 2, 3]
STAGE_NAMES  = {
    0: 'Healthy',
    1: 'Profit Reduction',
    2: 'Mild Liquidity Distress',
    3: 'Severe Liquidity Distress',
}

# ── Hyper-parameters for search
N_ITER       = 10          # RandomizedSearchCV iterations
RANDOM_STATE = 42
CV_SPLITS    = 5           # TimeSeriesSplit folds inside CV
TOP_K_MI     = 6           # Top-K features per binary flag (MI)
CORR_THRESH  = 0.98        # Correlation filter threshold
MISS_THRESH  = 0.30        # Missingness filter threshold



In [ ]:
# §2 — Mount Google Drive & Set Working Directory
# Reason: All intermediate and final artefacts saved here for reproducibility.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR        = '/content/drive/MyDrive/FYP'
RAW_FILE         = os.path.join(DRIVE_DIR, 'FYP.xlsx')
CHECKPOINT_FILE  = os.path.join(DRIVE_DIR, 'psx_clean_checkpoint.xlsx')
FEAT_CONFIG_FILE = os.path.join(DRIVE_DIR, 'psx_feature_config.json')
RESULTS_DIR      = os.path.join(DRIVE_DIR, 'results')

os.makedirs(DRIVE_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR,  exist_ok=True)
os.chdir(DRIVE_DIR)

print(f'Working directory : {os.getcwd()}')
print(f'Raw file          : {RAW_FILE}')
print(f'Checkpoint        : {CHECKPOINT_FILE}')
print(f'Results           : {RESULTS_DIR}')


In [ ]:
# §3 — Load Raw Data
# Reason: Excel header is on row index 1 (0-indexed). We rename to canonical
#         column names. Unnamed columns are Excel row-number artifacts — dropped.
df_raw = pd.read_excel(RAW_FILE, header=1)
print(f'Raw shape         : {df_raw.shape}')
print(f'Raw columns (first 10): {list(df_raw.columns[:10])}')

df = df_raw.copy()

# Drop any Unnamed artifact columns (index columns Excel sometimes writes)
unnamed = [c for c in df.columns if str(c).startswith('Unnamed')]
if unnamed:
    df = df.drop(columns=unnamed)
    print(f'Dropped Unnamed columns: {unnamed}')

print(f'Shape after drop  : {df.shape}')
print(f'Firms             : {df[FIRM_COL].nunique()}')
print(f'Year range        : {df[YEAR_COL].min()} – {df[YEAR_COL].max()}')
print(f'\nTarget distribution (Main_Distress):\n{df[MAIN_TARGET].value_counts(dropna=False).sort_index()}')


In [ ]:
# §4 — Target Engineering: Create T+1 Labels
#
# Design:
# - Sort by firm + year to ensure correct temporal ordering
# - Shift MAIN_TARGET and each binary flag by -1 (within firm) → next-year value
# - Healthy_T1 is derived from Main_Distress_T1 == 0 (no separate column needed)
# - Drop rows with missing Main_Distress_T1 (last year of each firm has no future)
# - Drop rows with missing Main_Distress (current-year label used as feature)
# - Drop duplicate (Firm, Year) pairs

# ── Sort
df = df.sort_values([FIRM_COL, YEAR_COL]).reset_index(drop=True)

# ── Create T+1 targets
df[TARGET_COL] = df.groupby(FIRM_COL)[MAIN_TARGET].shift(-1)
for col in BINARY_COLS:
    df[f'{col}_T1'] = df.groupby(FIRM_COL)[col].shift(-1)

print(f'Shape after T+1 shift : {df.shape}')

# ── Drop rows with missing Main_Distress_T1 (no future available)
n_before = len(df)
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)
print(f'Dropped (no T+1 target)         : {n_before - len(df)}')

# ── Drop rows with missing Main_Distress (needed as feature)
n_before = len(df)
df = df.dropna(subset=[MAIN_TARGET]).reset_index(drop=True)
print(f'Dropped (no Main_Distress label) : {n_before - len(df)}')

# ── Drop exact duplicates on (Firm, Year)
n_before = len(df)
df = df.drop_duplicates(subset=[FIRM_COL, YEAR_COL]).reset_index(drop=True)
print(f'Dropped duplicates               : {n_before - len(df)}')

# ── Cast target to int
df[TARGET_COL] = df[TARGET_COL].astype(int)

print(f'\nFinal shape : {df.shape}')
print(f'Year range  : {df[YEAR_COL].min()} – {df[YEAR_COL].max()}')
print(f'\nTarget distribution (Main_Distress_T1):')
dist = df[TARGET_COL].value_counts().sort_index()
for k, v in dist.items():
    pct = v / len(df) * 100
    print(f'  Stage {k} ({STAGE_NAMES[k]:<28}): {v:5d}  ({pct:.1f}%)')


In [ ]:
# §5 — Cleaning Pipeline
#
# Order matters:
# 1. Identify ratio feature candidates (numeric, not ID/target/binary columns)
# 2. Drop features with >30% missing values (computed on FULL cleaned dataset here;
#    re-applied on training fold in §6 for strict temporal isolation)
# 3. Correlation filter at 0.98: for each correlated pair drop the one with
#    lower mean MI (mutual information against Main_Distress_T1). Main_Distress is
#    always protected.
# 4. Save checkpoint with ALL rows — split happens in §6.

# ── Identify ratio candidates
excluded_always = (
    [FIRM_COL, YEAR_COL, MAIN_TARGET, TARGET_COL]
    + BINARY_COLS + T1_COLS
)
RATIO_COLS = [
    c for c in df.columns
    if c not in excluded_always and pd.api.types.is_numeric_dtype(df[c])
]
print(f'Ratio feature candidates: {len(RATIO_COLS)}')

# ── 5.1 Missingness filter (on full dataset for checkpoint; re-validated on train later)
miss_pct = df[RATIO_COLS].isnull().mean()
drop_miss = miss_pct[miss_pct > MISS_THRESH].index.tolist()
ratio_after_miss = [c for c in RATIO_COLS if c not in drop_miss]
print(f'\nMissingness filter (>{MISS_THRESH:.0%})  dropped: {len(drop_miss)} features')
if drop_miss:
    for f in drop_miss:
        print(f'  - {f} ({miss_pct[f]:.1%} missing)')
print(f'Remaining: {len(ratio_after_miss)} features')

# ── 5.2 Correlation filter
# Compute MI scores for all ratio candidates against TARGET_COL (for tie-breaking)
X_corr_mi = df[ratio_after_miss].fillna(df[ratio_after_miss].median())
mi_global  = pd.Series(
    mutual_info_classif(X_corr_mi, df[TARGET_COL].astype(int),
                        random_state=RANDOM_STATE),
    index=ratio_after_miss
)

corr_mat    = df[ratio_after_miss].corr().abs()
to_drop_corr = set()

for i in range(len(corr_mat.columns)):
    for j in range(i):
        if corr_mat.iloc[i, j] >= CORR_THRESH:
            fi = corr_mat.columns[i]
            fj = corr_mat.columns[j]
            # Protect Main_Distress (already excluded but guard anyway)
            if fi == MAIN_TARGET:
                to_drop_corr.add(fj)
            elif fj == MAIN_TARGET:
                to_drop_corr.add(fi)
            else:
                # Drop the one with LOWER MI against target
                drop_this = fi if mi_global.get(fi, 0) < mi_global.get(fj, 0) else fj
                to_drop_corr.add(drop_this)

ratio_after_corr = [c for c in ratio_after_miss if c not in to_drop_corr]
print(f'\nCorrelation filter (>{CORR_THRESH})      dropped: {len(to_drop_corr)} features')
if to_drop_corr:
    for f in sorted(to_drop_corr):
        print(f'  - {f}')
print(f'Remaining: {len(ratio_after_corr)} features')

# ── Visualize correlation heatmap of dropped features (subset)
corr_affected = sorted(
    set(to_drop_corr) |
    set(f for i in range(len(corr_mat.columns))
        for j in range(i)
        if corr_mat.iloc[i, j] >= CORR_THRESH
        for f in [corr_mat.columns[i], corr_mat.columns[j]])
)
if corr_affected:
    sub_corr = df[corr_affected].corr().abs()
    fig, ax = plt.subplots(figsize=(max(8, len(corr_affected)*0.6),
                                     max(6, len(corr_affected)*0.6)))
    sns.heatmap(sub_corr, annot=True, fmt='.2f', cmap='RdYlGn_r',
                linewidths=0.4, ax=ax, vmin=0, vmax=1)
    ax.set_title(f'Highly Correlated Features (≥{CORR_THRESH})', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, '01_correlation_heatmap.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

# ── 5.3 Store cleaned survivor list and save checkpoint
CLEAN_RATIO_COLS = ratio_after_corr
print(f'\nFinal clean ratio feature count: {len(CLEAN_RATIO_COLS)}')

# Save checkpoint: full data with clean columns only
save_cols = [FIRM_COL, YEAR_COL, MAIN_TARGET, TARGET_COL] + BINARY_COLS + T1_COLS + CLEAN_RATIO_COLS
df_ckpt   = df[save_cols].copy()
df_ckpt.to_excel(CHECKPOINT_FILE, index=False)
print(f'Checkpoint saved → {CHECKPOINT_FILE}')


In [ ]:
# §6 — Temporal Split: Last ~20% per Class for Test
#
# Why: This is panel/time-series data. A naive random split would leak future
# information into the training set. We must respect chronological ordering.
#
# Strategy: Within each class (Stage 0-3), take the chronologically LAST 20%
# of observations as the test set. This ensures each class is represented in
# test, and no future data is visible during training.
#
# This is superior to a single cutoff year because:
#   (a) minority classes (Stage 3) may have very few observations in a fixed year window
#   (b) guarantees ~20% per class balance in evaluation

df_ckpt = pd.read_excel(CHECKPOINT_FILE)
df_ckpt = df_ckpt.sort_values([FIRM_COL, YEAR_COL]).reset_index(drop=True)

train_idx_list = []
test_idx_list  = []

for cls in STAGE_LABELS:
    cls_idx = df_ckpt[df_ckpt[TARGET_COL] == cls].index.tolist()
    if len(cls_idx) == 0:
        print(f'  WARNING: Class {cls} has 0 samples — skipped')
        continue
    n_test  = max(1, int(len(cls_idx) * 0.20))
    # Last n_test observations (chronologically) → test
    test_idx_list.extend(cls_idx[-n_test:])
    train_idx_list.extend(cls_idx[:-n_test])

train_df = df_ckpt.loc[train_idx_list].sort_values([FIRM_COL, YEAR_COL]).reset_index(drop=True)
test_df  = df_ckpt.loc[test_idx_list ].sort_values([FIRM_COL, YEAR_COL]).reset_index(drop=True)

print(f'Train : {train_df.shape}  | years {train_df[YEAR_COL].min()}–{train_df[YEAR_COL].max()}')
print(f'Test  : {test_df.shape}   | years {test_df[YEAR_COL].min()}–{test_df[YEAR_COL].max()}')
print(f'Test % overall: {len(test_df)/(len(train_df)+len(test_df)):.1%}')

print('\nTrain class distribution:')
for k, v in train_df[TARGET_COL].value_counts().sort_index().items():
    print(f'  Stage {int(k)} ({STAGE_NAMES[int(k)]:<28}): {v}  ({v/len(train_df)*100:.1f}%)')

print('\nTest class distribution:')
for k, v in test_df[TARGET_COL].value_counts().sort_index().items():
    print(f'  Stage {int(k)} ({STAGE_NAMES[int(k)]:<28}): {v}  ({v/len(test_df)*100:.1f}%)')

# ── Re-validate missingness filter on TRAINING DATA ONLY
miss_train = train_df[CLEAN_RATIO_COLS].isnull().mean()
drop_miss_train = miss_train[miss_train > MISS_THRESH].index.tolist()
if drop_miss_train:
    for f in drop_miss_train:
        print(f'   {f} ({miss_train[f]:.1%})')
    CLEAN_RATIO_COLS = [c for c in CLEAN_RATIO_COLS if c not in drop_miss_train]

print(f'\nFinal clean ratio features (train-validated): {len(CLEAN_RATIO_COLS)}')


In [ ]:
# §7 — Custom Transformers: Firm-Wise Rolling Imputer & Preprocessing Pipeline

class FirmWiseRollingImputer(BaseEstimator, TransformerMixin):
    """
    Impute missing values using firm-specific rolling mean on past observations.
    The 'window' parameter controls how many past periods to include.
    min_periods=1 ensures we get a value even for the very first observation.
    """
    def __init__(self, window=5, firm_col=FIRM_COL, year_col=YEAR_COL):
        self.window   = window
        self.firm_col = firm_col
        self.year_col = year_col

    def fit(self, X, y=None):
        self.feature_cols_ = [c for c in X.columns if c not in [self.firm_col, self.year_col]]
        self.global_medians_ = X[self.feature_cols_].median()

        # Store the last 'window' non-NaN values for each feature per firm
        # from the training data X. This will be used to initialize rolling
        # calculations in transform() for firms seen in fit().
        self.firm_last_values_ = {}

        # Ensure X is sorted for correct temporal processing
        X_sorted = X.sort_values([self.firm_col, self.year_col])

        for firm, firm_df in X_sorted.groupby(self.firm_col):
            self.firm_last_values_[firm] = {}
            for col in self.feature_cols_:
                # Get the last `window` non-NaN values for this firm and column
                # These are the actual observations that would precede any new data
                last_values = firm_df[col].dropna().tail(self.window).values
                self.firm_last_values_[firm][col] = last_values

        return self

    def transform(self, X):
        Xt = X.copy()
        # Sort X to ensure correct temporal order for rolling operations
        # and to group by firm for efficient processing
        Xt_sorted = Xt.sort_values([self.firm_col, self.year_col])

        transformed_dfs = []

        for firm, firm_df in Xt_sorted.groupby(self.firm_col):
            current_firm_df = firm_df.copy() # Operate on a copy of the group's dataframe

            for col in self.feature_cols_:
                if col not in current_firm_df.columns:
                    # Skip if the feature column is not in the current firm's dataframe
                    # (e.g., if X has fewer columns than feature_cols_ defined in fit)
                    continue

                # Get the stored last values from training data for this firm and column
                # If firm not seen in fit, or column not in history, use empty array
                history_values = self.firm_last_values_.get(firm, {}).get(col, np.array([]))

                # Combine historical values with current data for correct rolling imputation
                # The combined series will have the historical values followed by current values
                combined_series_data = np.concatenate([history_values, current_firm_df[col].values])
                combined_series = pd.Series(combined_series_data)

                # Calculate the rolling mean using the combined series.
                # `shift(1)` ensures only past values are used.
                # `min_periods=1` ensures imputation for early values.
                rolling_means = combined_series.shift(1).rolling(
                    window=self.window, min_periods=1
                ).mean()

                # The `rolling_means` series has values for both history and current data.
                # We only need the part corresponding to the current firm_df.
                # This starts after the historical values.
                imputation_values = rolling_means.iloc[len(history_values):].values

                # Fill NaNs in the current column of current_firm_df with the imputation_values
                # Use `current_firm_df.index` to ensure alignment if indices are non-consecutive
                current_firm_df[col] = current_firm_df[col].fillna(
                    pd.Series(imputation_values, index=current_firm_df.index)
                )

                # Fill any remaining NaNs (e.g., first observations of a new firm in transform,
                # or if all historical/current values are NaN) with the global median from fit().
                current_firm_df[col] = current_firm_df[col].fillna(
                    self.global_medians_.get(col, 0)
                )
            transformed_dfs.append(current_firm_df)

        # Concatenate all processed firm dataframes.
        # Use reindex to restore the original order of the input `Xt` (before sorting).
        if transformed_dfs:
            Xt_final = pd.concat(transformed_dfs).reindex(Xt.index)
        else:
            # If transformed_dfs is empty (e.g., if Xt was empty), just return a copy with NaNs filled by global median
            Xt_final = Xt.copy()
            for col in self.feature_cols_:
                Xt_final[col] = Xt_final[col].fillna(self.global_medians_.get(col, 0))

        return Xt_final


class MedianFallbackImputer(BaseEstimator, TransformerMixin):
    """Global median fallback for any NaN remaining after firm-wise imputation."""
    def fit(self, X, y=None):
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        self.medians_ = X[numeric_cols].median()
        return self

    def transform(self, X):
        Xt = X.copy()
        for col, med in self.medians_.items():
            if col in Xt.columns:
                Xt[col] = Xt[col].fillna(med)
        return Xt


class WinsorizerTransformer(BaseEstimator, TransformerMixin):
    """Clip each feature to [1st, 99th] percentile computed on training data."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        self._bounds = {
            col: (X[col].quantile(self.lower), X[col].quantile(self.upper))
            for col in numeric_cols
        }
        return self

    def transform(self, X):
        Xt = X.copy()
        for col, (lo, hi) in self._bounds.items():
            if col in Xt.columns:
                Xt[col] = Xt[col].clip(lower=lo, upper=hi)
        return Xt


class DropIdColumns(BaseEstimator, TransformerMixin):
    """Drop identifier columns (Firm, Year) before scaling."""
    def __init__(self, cols=None):
        self.cols = cols or [FIRM_COL, YEAR_COL]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=[c for c in self.cols if c in X.columns])


def build_preprocessing_pipeline():
    return Pipeline([
        ('firm_imputer',    FirmWiseRollingImputer(window=5)),
        ('median_fallback', MedianFallbackImputer()),
        ('winsorizer',      WinsorizerTransformer()),
        ('drop_ids',        DropIdColumns()),
        ('scaler',          RobustScaler()),
    ])


def build_full_pipeline(clf, use_smote=False):
    """
    Full pipeline: preprocessing → optional SMOTE → classifier.
    SMOTE is placed AFTER preprocessing so it only augments scaled,
    imputed data → never raw or unscaled features.
    """
    steps = list(build_preprocessing_pipeline().steps)
    if use_smote:
        steps += [('smote', SMOTE(sampling_strategy='not majority',
                                   random_state=RANDOM_STATE, k_neighbors=5))]
    steps += [('clf', clf)]
    return (ImbPipeline if use_smote else Pipeline)(steps=steps)


print('Pipeline preview:', [s for s, _ in build_full_pipeline(DecisionTreeClassifier()).steps])


In [ ]:
# §8 — MI Feature Selection: Top-6 per Binary Flag (1vsRest)
#
# Strategy:
# 1. Create 4 binary "1-vs-rest" flags:
#    - Profit_Reduction_T1 vs rest   (Stage 1 vs {0,2,3})
#    - Mild_Liquidity_Distress_T1    (Stage 2 vs {0,1,3})
#    - Severe_Liquidity_Distress_T1  (Stage 3 vs {0,1,2})
#    - Healthy_T1                    (Stage 0 vs {1,2,3})
#
# 2. Compute MI for each flag separately → take Top-6 per flag
# 3. Union of all selected features
# 4. ALWAYS add Main_Distress explicitly (strong autocorrelation signal;
#    firms in distress tend to remain — up to 79% persistence in Stage 3)
#
# Why Top-6 instead of Top-5?
# With 4 flags instead of 3, the union can be slightly smaller unless we
# increase K. Top-6 gives ~15-20 features in the union — practical for
# the small dataset size without overfitting.
#
# Why union instead of intersection?
# Different distress stages have different financial signatures. Mild liquidity
# distress is detected by current-ratio-type features; severe distress by
# leverage ratios. Intersection would throw away these stage-specific signals.

# ── Derive Healthy_T1
train_df['Healthy_T1'] = (train_df[TARGET_COL] == 0).astype(int)

ALL_T1_FLAGS = T1_COLS + ['Healthy_T1']  # 4 binary flags

# ── MI candidates: clean ratio cols only (no identifiers, no targets)
mi_candidates = [
    c for c in CLEAN_RATIO_COLS
    if pd.api.types.is_numeric_dtype(train_df[c]) and c != MAIN_TARGET
]
X_mi = train_df[mi_candidates].fillna(train_df[mi_candidates].median())

selected_per_flag  = {}
mi_scores_per_flag = {}

print(f'MI candidates: {len(mi_candidates)} features')
print(f'Flags: {ALL_T1_FLAGS}')
print()

for t1_col in ALL_T1_FLAGS:
    y_mi = train_df[t1_col]
    valid = y_mi.notna() & y_mi.isin([0, 1])
    if valid.sum() < 30 or y_mi[valid].nunique() < 2:
        print(f'  SKIP {t1_col} (insufficient data)')
        continue
    mi = mutual_info_classif(
        X_mi[valid], y_mi[valid].astype(int),
        random_state=RANDOM_STATE, n_neighbors=5
    )
    mi_s = pd.Series(mi, index=mi_candidates).sort_values(ascending=False)
    mi_scores_per_flag[t1_col] = mi_s
    selected_per_flag[t1_col]  = mi_s.head(TOP_K_MI).index.tolist()
    print(f'{t1_col:<38} top-{TOP_K_MI}: {selected_per_flag[t1_col]}')

# ── Union
union_feats = sorted(set(f for fl in selected_per_flag.values() for f in fl))

# ── Always add Main_Distress
if MAIN_TARGET not in union_feats:
    union_feats = [MAIN_TARGET] + union_feats

SELECTED_FEATURES = union_feats

print(f'\nUnion of top-{TOP_K_MI} per flag : {len(union_feats) - 1} features')
print(f'+ Main_Distress always added')
print(f'Total selected features        : {len(SELECTED_FEATURES)}')
print(f'\nFinal feature list:')
for i, f in enumerate(SELECTED_FEATURES):
    print(f'  {i+1:2d}. {f}')

# ── Save feature config
feat_config = {
    'selected_features':   SELECTED_FEATURES,
    'selected_per_flag':   selected_per_flag,
    'clean_ratio_cols':    CLEAN_RATIO_COLS,
    'top_k_mi':            TOP_K_MI,
    'corr_thresh':         CORR_THRESH,
    'miss_thresh':         MISS_THRESH,
}
with open(FEAT_CONFIG_FILE, 'w') as f:
    json.dump(feat_config, f, indent=2)
print(f'\nFeature config saved → {FEAT_CONFIG_FILE}')


In [ ]:
# §8b — Derive Layer-Specific Feature Sets for Hierarchical Strategy
#
# WHY separate feature sets per layer?
# Layer 2a (Stage 0 vs Stage 1) should focus on features that distinguish
# healthy firms from mildly distressed ones — profitability ratios dominate.
# Layer 2b (Stage 2 vs Stage 3) should focus on features distinguishing mild
# vs severe liquidity distress — leverage and coverage ratios dominate.
# Using the full SMOTE feature union in sub-layers adds noise from unrelated
# distress stages, which hurts binary sub-classifiers.

# ── Features for Layer 1: {0,1} vs {2,3}
# Use union of (Stage 0 + Stage 1) features AND (Stage 2 + Stage 3) features
# i.e., the full SELECTED_FEATURES is appropriate here
FEATS_L1     = SELECTED_FEATURES   # Full set for coarse separation

# ── Features for Layer 2a: Stage 0 vs Stage 1
# Top-6 for Healthy_T1 + Top-6 for Profit_Reduction_T1 + Main_Distress
feats_2a_raw = (
    selected_per_flag.get('Healthy_T1', []) +
    selected_per_flag.get('Profit_Reduction_T1', [])
)
feats_2a_union = sorted(set(feats_2a_raw))
if MAIN_TARGET not in feats_2a_union:
    feats_2a_union = [MAIN_TARGET] + feats_2a_union
FEATS_L2A = feats_2a_union

# ── Features for Layer 2b: Stage 2 vs Stage 3
feats_2b_raw = (
    selected_per_flag.get('Mild_Liquidity_Distress_T1', []) +
    selected_per_flag.get('Severe_Liquidity_Distress_T1', [])
)
feats_2b_union = sorted(set(feats_2b_raw))
if MAIN_TARGET not in feats_2b_union:
    feats_2b_union = [MAIN_TARGET] + feats_2b_union
FEATS_L2B = feats_2b_union

print('Feature set sizes:')
print(f'  SMOTE / Layer 1       : {len(FEATS_L1)} features')
print(f'  Layer 2a (Stage 0v1)  : {len(FEATS_L2A)} features  → {FEATS_L2A}')
print(f'  Layer 2b (Stage 2v3)  : {len(FEATS_L2B)} features  → {FEATS_L2B}')

# ── Build X/y matrices
ID_COLS = [FIRM_COL, YEAR_COL]

X_train_full = train_df[SELECTED_FEATURES + ID_COLS].copy()
y_train      = train_df[TARGET_COL].astype(int).reset_index(drop=True)
X_test_full  = test_df[SELECTED_FEATURES  + ID_COLS].copy()
y_test       = test_df[TARGET_COL].astype(int).reset_index(drop=True)

print(f'\nX_train : {X_train_full.shape}')
print(f'X_test  : {X_test_full.shape}')


In [ ]:
# §8c — MI Feature Importance Visualisation
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (t1_col, mi_s) in enumerate(mi_scores_per_flag.items()):
    ax = axes[i]
    top = mi_s.head(TOP_K_MI)
    colors = ['#1565C0' if f != MAIN_TARGET else '#E53935' for f in top.index]
    top.plot(kind='barh', ax=ax, color=colors[::-1])
    ax.invert_yaxis()
    ax.set_title(f'Top-{TOP_K_MI}: {t1_col}', fontsize=10, fontweight='bold')
    ax.set_xlabel('MI Score')
    for j, (v, name) in enumerate(zip(top.values[::-1], top.index[::-1])):
        ax.text(v + 0.001, j, f'{v:.4f}', va='center', fontsize=8)

for j in range(i+1, 4):
    axes[j].set_visible(False)

plt.suptitle(f'Mutual Information — Top-{TOP_K_MI} Features per Binary Flag (1-vs-Rest)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '02_mi_feature_selection.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('MI visualisation saved.')


In [ ]:
# §9 — Model Configurations
#
# Selection rationale:
# TREE MODELS (3):
#   - RandomForest     : Bagged decision trees; high variance reduction; well-calibrated
#                        with balanced_subsample for imbalanced data
#   - ExtraTrees       : Further randomises split thresholds → lower variance than RF,
#                        faster training; excellent on tabular financial data
#   - DecisionTree     : Single tree as interpretable baseline; also useful for SHAP
#
# ENSEMBLE MODELS (3):
#   - LightGBM         : Gradient boosting with leaf-wise growth; state-of-the-art on
#                        tabular data; handles missing values natively; very fast
#   - XGBoost          : Regularised gradient boosting; level-wise growth for
#                        generalization; complementary to LightGBM
#   - BaggingClassifier: Bootstrap aggregating over DT; explicit diversity mechanism;
#                        reduces variance without bias; good for small datasets
#
# NOT included (reasons):
#   - LogisticRegression: Cannot capture non-linear interactions in financial ratios
#   - Naive Bayes      : Feature independence assumption violated by financial ratios
#   - KNN              : O(n) prediction cost; noisy in high-dimensional feature space
#   - SVM              : Does not scale well with N_ITER; poor with imbalanced classes
#   - AdaBoost         : Sensitive to outliers even after winsorization; dominated by LGBM/XGB
#   - HistGradientBoosting: Redundant with LightGBM for this dataset size

MODEL_CONFIGS = {
    # ── Tree Models
    'RandomForest': {
        'estimator': RandomForestClassifier(
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE, n_jobs=-1),
        'params': {
            'clf__n_estimators':     [100, 200, 300, 500],
            'clf__max_depth':        [3, 5, 7, 10, None],
            'clf__min_samples_leaf': [3, 5, 10, 20],
            'clf__max_features':     ['sqrt', 'log2', 0.5],
            'clf__min_samples_split':[2, 5, 10],
        }
    },
    'ExtraTrees': {
        'estimator': ExtraTreesClassifier(
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE, n_jobs=-1),
        'params': {
            'clf__n_estimators':     [100, 200, 300, 500],
            'clf__max_depth':        [3, 5, 7, 10, None],
            'clf__min_samples_leaf': [3, 5, 10, 20],
            'clf__max_features':     ['sqrt', 'log2', 0.5],
        }
    },
    'DecisionTree': {
        'estimator': DecisionTreeClassifier(
            class_weight='balanced', random_state=RANDOM_STATE),
        'params': {
            'clf__max_depth':        [3, 5, 7, 10, None],
            'clf__min_samples_leaf': [3, 5, 10, 20],
            'clf__max_features':     ['sqrt', 'log2', None],
            'clf__criterion':        ['gini', 'entropy'],
            'clf__min_samples_split':[2, 5, 10],
        }
    },
    # ── Ensemble Models
    'LightGBM': {
        'estimator': LGBMClassifier(
            class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
        'params': {
            'clf__n_estimators':      [100, 200, 300, 500],
            'clf__learning_rate':     [0.01, 0.03, 0.05, 0.1],
            'clf__max_depth':         [3, 5, 7, -1],
            'clf__num_leaves':        [15, 31, 50, 63],
            'clf__min_child_samples': [5, 10, 20, 30],
            'clf__reg_alpha':         [0, 0.1, 1.0],
            'clf__reg_lambda':        [0, 0.1, 1.0],
        }
    },
    'XGBoost': {
        'estimator': XGBClassifier(
            eval_metric='mlogloss',
            random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
        'params': {
            'clf__n_estimators':     [100, 200, 300, 500],
            'clf__learning_rate':    [0.01, 0.03, 0.05, 0.1],
            'clf__max_depth':        [3, 5, 7],
            'clf__reg_alpha':        [0, 0.1, 1.0],
            'clf__reg_lambda':       [0.1, 1.0, 5.0],
            'clf__subsample':        [0.6, 0.8, 1.0],
            'clf__colsample_bytree': [0.6, 0.8, 1.0],
        }
    },
    'Bagging': {
        'estimator': BaggingClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=7, class_weight='balanced',
                random_state=RANDOM_STATE),
            random_state=RANDOM_STATE, n_jobs=-1),
        'params': {
            'clf__n_estimators': [50, 100, 200, 300],
            'clf__max_samples':  [0.5, 0.7, 0.8, 1.0],
            'clf__max_features': [0.7, 0.8, 1.0],
            'clf__bootstrap':    [True, False],
        }
    },
}

for name, cfg in MODEL_CONFIGS.items():
    print(f'  - {name}  ({type(cfg["estimator"]).__name__})')


In [ ]:
# §10 — Evaluation Framework
#
# Primary metric: Macro Recall
#   - Equally weights all 4 stages, critical for early-warning systems where
#     missing any stage is costly. Precision matters less than recall here.
# Secondary metrics: Stage 3 Recall (most severe), Stage 1 Recall (hardest to detect)
# All CV uses TimeSeriesSplit to preserve temporal ordering within folds.

def compute_metrics(y_true, y_pred, prefix=''):
    """Compute standard classification metrics."""
    return {
        f'{prefix}Macro Recall':     recall_score(y_true, y_pred, average='macro',    zero_division=0),
        f'{prefix}Macro Precision':  precision_score(y_true, y_pred, average='macro', zero_division=0),
        f'{prefix}Macro F1':         f1_score(y_true, y_pred, average='macro',        zero_division=0),
        f'{prefix}Balanced Acc':     balanced_accuracy_score(y_true, y_pred),
        f'{prefix}Accuracy':         accuracy_score(y_true, y_pred),
    }


def classwise_recall(y_true, y_pred, labels=STAGE_LABELS):
    rows = []
    for lbl in labels:
        yt = (np.array(y_true) == lbl).astype(int)
        yp = (np.array(y_pred) == lbl).astype(int)
        rows.append({
            'Stage':     lbl,
            'Name':      STAGE_NAMES[lbl],
            'Recall':    recall_score(yt, yp, zero_division=0),
            'Precision': precision_score(yt, yp, zero_division=0),
            'F1':        f1_score(yt, yp, zero_division=0),
            'Support':   int((np.array(y_true) == lbl).sum()),
        })
    return pd.DataFrame(rows).set_index('Stage')


def evaluate_model(model, X_tr, y_tr, X_te, y_te, label='Model'):
    y_tr_p = model.predict(X_tr)
    y_te_p = model.predict(X_te)
    row    = {'Model': label}
    row.update(compute_metrics(y_tr, y_tr_p, 'Train '))
    row.update(compute_metrics(y_te, y_te_p, 'Test '))
    cwr = classwise_recall(y_te, y_te_p)
    for s in STAGE_LABELS:
        row[f'Stage {s} Recall'] = cwr.loc[s, 'Recall'] if s in cwr.index else 0.0
    row['Train-Test Gap'] = row['Train Macro Recall'] - row['Test Macro Recall']
    return row, y_tr_p, y_te_p


def run_experiment(name, X_tr, y_tr, X_te, y_te,
                   configs=None, use_smote=False,
                   n_iter=N_ITER, scoring='recall_macro'):
    """
    Run RandomizedSearchCV for all model configs.
    TimeSeriesSplit(CV_SPLITS) preserves temporal ordering in CV folds.
    Returns ranked results DataFrame and best fitted model.
    """
    configs = configs or MODEL_CONFIGS
    cv      = TimeSeriesSplit(n_splits=CV_SPLITS)
    rows, fitted = [], {}

    print(f'\n{"═"*70}')
    print(f'  {name}')
    print(f'{'─'*70}')

    for mname, cfg in configs.items():
        try:
            pipe = build_full_pipeline(clone(cfg['estimator']), use_smote=use_smote)
            search = RandomizedSearchCV(
                pipe, cfg['params'],
                n_iter=n_iter, scoring=scoring, cv=cv,
                random_state=RANDOM_STATE, n_jobs=-1,
                refit=True, verbose=0, error_score='raise',
            )
            t0 = time.time()
            search.fit(X_tr, y_tr)
            elapsed = time.time() - t0
            best_pipe = search.best_estimator_
            row, _, _ = evaluate_model(best_pipe, X_tr, y_tr, X_te, y_te, label=mname)
            row['Best Params'] = str(search.best_params_)
            row['CV Score']    = round(search.best_score_, 4)
            row['Time (s)']    = round(elapsed, 1)
            rows.append(row)
            fitted[mname] = best_pipe
                  f' | S1: {row["Stage 1 Recall"]:.4f}'
                  f' | S3: {row["Stage 3 Recall"]:.4f}'
                  f' | CV: {row["CV Score"]:.4f}'
                  f' | {elapsed:.0f}s')
        except Exception as e:

    df_res = pd.DataFrame(rows)
    if not df_res.empty:
        df_res = df_res.sort_values(
            ['Test Macro Recall', 'Stage 3 Recall', 'Train-Test Gap'],
            ascending=[False, False, True]
        ).reset_index(drop=True)

    best_name  = df_res.iloc[0]['Model'] if not df_res.empty else None
    best_model = fitted.get(best_name)
    return df_res, best_name, best_model, fitted


DISPLAY_COLS = [
    'Model', 'Train Macro Recall', 'Test Macro Recall',
    'Test Macro F1', 'Test Balanced Acc',
    'Stage 0 Recall', 'Stage 1 Recall', 'Stage 2 Recall', 'Stage 3 Recall',
    'Train-Test Gap', 'CV Score',
]



In [ ]:
# §11 — Experiment A: SMOTE (Data-Level Balancing)
#
# SMOTE is placed inside the pipeline AFTER preprocessing:
#   preprocessing → SMOTE → classifier
# This ensures:
# 1. SMOTE only sees scaled, imputed, winsorized data (no raw features)
# 2. SMOTE is fitted only on training folds — test data is NEVER augmented
# 3. Synthetic samples are generated in the transformed feature space
#    (RobustScaler output), which is more meaningful than raw ratio space
#
# sampling_strategy='not majority': oversample all minority classes to
# match the majority class (Stage 0 = Healthy), preserving majority count.

results_smote, best_name_smote, best_model_smote, all_models_smote = run_experiment(
    name       = 'Experiment A — SMOTE (4-class flat classification)',
    X_tr       = X_train_full,
    y_tr       = y_train,
    X_te       = X_test_full,
    y_te       = y_test,
    use_smote  = True,
    n_iter     = N_ITER,
    scoring    = 'recall_macro',
)

print('\n— Experiment A Ranked Results —')
display(results_smote[DISPLAY_COLS].round(4))


In [ ]:
# §12 — Experiment B: Hierarchical Decomposition Strategy 1
#
# Architecture:
#   Layer 1  :  {Stage 0, Stage 1}  vs  {Stage 2, Stage 3}    [binary]
#                      ↓ predicted 0              ↓ predicted 1
#               Layer 2a                    Layer 2b
#               Stage 0 vs Stage 1          Stage 2 vs Stage 3
#               (uses FEATS_L2A)            (uses FEATS_L2B)
#
# Key design decisions:
# 1. Layer-specific features: each sub-problem uses only its relevant features
#    (Stage 0/1 features for 2a; Stage 2/3 features for 2b)
# 2. Exposure-bias fix: Layer 2 models are trained on samples that LAYER 1
#    PREDICTS belong to their group, not on true-label subsets. This exposes
#    sub-models to the noise/misrouting they will see at inference time.
# 3. Misrouting mapping:
#    - Layer 2a: misrouted Stage 2/3 → mapped to Stage 1 (mildest available)
#    - Layer 2b: misrouted Stage 0/1 → mapped to Stage 2 (mildest available)
# 4. No SMOTE in hierarchical layers: each binary layer already has a more
#    balanced distribution; SMOTE can distort sub-class boundaries.

class HierarchicalPredictor:
    """
    Two-layer sequential classifier.
    Layer 1 routes samples to groups.
    Layer 2 models refine within each group.
    label_map: maps Layer 2 binary outputs back to original labels.
    """
    def __init__(self, layer1_model, layer2_models, label_maps, feats_l2):
        self.layer1_model  = layer1_model
        self.layer2_models = layer2_models   # {grp_id: fitted_model}
        self.label_maps    = label_maps      # {grp_id: {binary_pred: original_label}}
        self.feats_l2      = feats_l2        # {grp_id: feature_list}

    def predict(self, X):
        X      = X.reset_index(drop=True) if hasattr(X, 'reset_index') else X
        l1_pred = self.layer1_model.predict(X)
        final   = np.full(len(X), 0, dtype=int)

        for grp in np.unique(l1_pred):
            mask  = np.where(l1_pred == grp)[0]
            X_sub = X.iloc[mask]

            # Select only the features this layer needs
            feats = self.feats_l2.get(grp, None)
            if feats is not None:
                id_cols_present = [c for c in [FIRM_COL, YEAR_COL] if c in X_sub.columns]
                feat_cols = [c for c in feats if c in X_sub.columns]
                X_sub_feat = X_sub[feat_cols + id_cols_present]
            else:
                X_sub_feat = X_sub

            if grp in self.layer2_models:
                raw_pred = self.layer2_models[grp].predict(X_sub_feat.reset_index(drop=True))
                # Map binary prediction back to original stage labels
                lmap = self.label_maps.get(grp, {})
                mapped = np.array([lmap.get(int(p), int(p)) for p in raw_pred])
                final[mask] = mapped
            else:
                # Default map (e.g., group with single class)
                lmap = self.label_maps.get(grp, {})
                final[mask] = lmap.get(grp, grp)

        return final


def train_best_layer_model(X_tr, y_tr, feat_list, configs=MODEL_CONFIGS,
                            n_iter=N_ITER, use_smote=False, layer_name='Layer'):
    """
    Select best model for a single layer using temporal validation (last 20%).
    X_tr should include ID cols; feat_list is the feature subset.
    """
    id_cols_present = [c for c in [FIRM_COL, YEAR_COL] if c in X_tr.columns]
    feat_cols       = [c for c in feat_list if c in X_tr.columns]
    X_layer = X_tr[feat_cols + id_cols_present].reset_index(drop=True)
    y_layer = y_tr.reset_index(drop=True)

    cv         = TimeSeriesSplit(n_splits=CV_SPLITS)
    split_idx  = int(len(X_layer) * 0.80)
    X_inner    = X_layer.iloc[:split_idx].reset_index(drop=True)
    y_inner    = y_layer.iloc[:split_idx].reset_index(drop=True)
    X_val      = X_layer.iloc[split_idx:].reset_index(drop=True)
    y_val      = y_layer.iloc[split_idx:].reset_index(drop=True)

    best_score  = -1
    best_mname  = None
    best_params = {}

    for mname, cfg in configs.items():
        try:
            pipe   = build_full_pipeline(clone(cfg['estimator']), use_smote=use_smote)
            search = RandomizedSearchCV(
                pipe, cfg['params'], n_iter=n_iter,
                scoring='recall_macro', cv=cv,
                random_state=RANDOM_STATE, n_jobs=-1,
                refit=True, verbose=0, error_score='raise',
            )
            search.fit(X_inner, y_inner)
            if len(y_val) > 0 and y_val.nunique() > 1:
                score = recall_score(y_val, search.predict(X_val),
                                     average='macro', zero_division=0)
            else:
                score = search.best_score_
            if score > best_score:
                best_score  = score
                best_mname  = mname
                best_params = search.best_params_
        except Exception as e:
            pass

    if best_mname is None:
        raise RuntimeError(f'{layer_name}: all configs failed')

    # Refit on full layer training data
    final_pipe = build_full_pipeline(
        clone(configs[best_mname]['estimator']), use_smote=use_smote)
    valid_params = {k: v for k, v in best_params.items()
                    if k in final_pipe.get_params()}
    if valid_params:
        final_pipe.set_params(**valid_params)
    final_pipe.fit(X_layer, y_layer)

    print(f'  {layer_name:<42} best={best_mname}  val_recall={best_score:.4f}  n={len(X_layer)}')
    return final_pipe, best_mname


# ─────────────────────────────────────────────────────────────────
print('═'*70)
print('  EXPERIMENT B — Hierarchical Strategy 1')
print('  Layer 1  : {0,1} vs {2,3}')
print('  Layer 2a : Stage 0 vs Stage 1   (features: Stage 0/1 specific)')
print('  Layer 2b : Stage 2 vs Stage 3   (features: Stage 2/3 specific)')
print('═'*70)

# ── Layer 1: binary coarse separation
id_cols_present_train = [c for c in [FIRM_COL, YEAR_COL] if c in X_train_full.columns]
y_tr_l1 = y_train.map({0: 0, 1: 0, 2: 1, 3: 1})
print(f'\nLayer 1 class dist: {y_tr_l1.value_counts().to_dict()}  (0=low, 1=high)')

model_l1, name_l1 = train_best_layer_model(
    X_train_full, y_tr_l1, FEATS_L1, layer_name='Layer 1 ({0,1} vs {2,3})')

# ── Route training data through Layer 1
feat_cols_l1     = [c for c in FEATS_L1 if c in X_train_full.columns]
l1_pred_train    = model_l1.predict(
    X_train_full[feat_cols_l1 + id_cols_present_train].reset_index(drop=True))

# ── Layer 2a — Stage 0 vs Stage 1 (on L1-predicted 0 group)
mask_low = np.where(l1_pred_train == 0)[0]
X_2a     = X_train_full.iloc[mask_low].reset_index(drop=True)
y_2a_raw = y_train.iloc[mask_low].reset_index(drop=True)
# Misrouted Stage 2/3 → mapped to Stage 1 (nearest in this sub-problem)
y_2a     = y_2a_raw.map({0: 0, 1: 1, 2: 1, 3: 1})

print(f'\nLayer 2a — {len(X_2a)} samples  true dist: {y_2a_raw.value_counts().to_dict()}')
print(f'  After mapping: {y_2a.value_counts().to_dict()}')

model_l2a, name_l2a = train_best_layer_model(
    X_2a, y_2a, FEATS_L2A, layer_name='Layer 2a (Stage 0 vs 1)')

# ── Layer 2b — Stage 2 vs Stage 3 (on L1-predicted 1 group)
mask_high = np.where(l1_pred_train == 1)[0]
X_2b      = X_train_full.iloc[mask_high].reset_index(drop=True)
y_2b_raw  = y_train.iloc[mask_high].reset_index(drop=True)
# Misrouted Stage 0/1 → mapped to Stage 2 (nearest in this sub-problem)
y_2b      = y_2b_raw.map({0: 2, 1: 2, 2: 2, 3: 3})

print(f'\nLayer 2b — {len(X_2b)} samples  true dist: {y_2b_raw.value_counts().to_dict()}')
print(f'  After mapping: {y_2b.value_counts().to_dict()}')

model_l2b, name_l2b = train_best_layer_model(
    X_2b, y_2b, FEATS_L2B, layer_name='Layer 2b (Stage 2 vs 3)')

# ── Assemble HierarchicalPredictor
hier_s1 = HierarchicalPredictor(
    layer1_model  = model_l1,
    layer2_models = {0: model_l2a, 1: model_l2b},
    label_maps    = {0: {0: 0, 1: 1},   # binary 0→Stage0, 1→Stage1
                     1: {0: 2, 1: 3}},  # binary 0→Stage2, 1→Stage3
    feats_l2      = {0: FEATS_L2A, 1: FEATS_L2B},
)

# ── Evaluate on test set (pass full X with ID cols; model selects internally)
y_pred_hier_tr = hier_s1.predict(X_train_full)
y_pred_hier_te = hier_s1.predict(X_test_full)

tr_h = compute_metrics(y_train, y_pred_hier_tr, 'Train ')
te_h = compute_metrics(y_test,  y_pred_hier_te, 'Test ')
cw_h = classwise_recall(y_test, y_pred_hier_te)

results_hier = pd.DataFrame([{
    'Model':         'Hierarchical_S1',
    **tr_h, **te_h,
    **{f'Stage {s} Recall': cw_h.loc[s, 'Recall'] if s in cw_h.index else 0.0
       for s in STAGE_LABELS},
    'Train-Test Gap': tr_h['Train Macro Recall'] - te_h['Test Macro Recall'],
    'CV Score': float('nan'),
    'Best Params': f'L1={name_l1}, L2a={name_l2a}, L2b={name_l2b}',
    'Time (s)': float('nan'),
}])
best_model_hier = hier_s1
best_name_hier  = 'Hierarchical_S1'

print('\n— Experiment B Results —')
display(results_hier[DISPLAY_COLS].round(4))
print('\nClass-wise Recall (Test):')
display(cw_h.round(4))


In [ ]:
# §13 — Master Comparison Table: SMOTE vs Hierarchical
#
# Best model from each experiment is selected by Test Macro Recall.
# Overall best is then identified across both experiments.

# ── Row for Experiment A (SMOTE best model)
row_smote = results_smote.iloc[0].to_dict()
row_smote['Experiment'] = 'Exp A — SMOTE'
row_smote['Paradigm']   = 'Data-Level Balancing'

# ── Row for Experiment B (Hierarchical)
row_hier        = results_hier.iloc[0].to_dict()
row_hier['Experiment'] = 'Exp B — Hierarchical S1'
row_hier['Paradigm']   = 'Decomposition'

master_df = pd.DataFrame([row_smote, row_hier])

MASTER_COLS = [
    'Experiment', 'Model', 'Paradigm',
    'Train Macro Recall', 'Test Macro Recall',
    'Test Macro F1', 'Test Balanced Acc',
    'Stage 0 Recall', 'Stage 1 Recall', 'Stage 2 Recall', 'Stage 3 Recall',
    'Train-Test Gap',
]

print('\n' + '═'*72)
print('  MASTER COMPARISON — SMOTE vs Hierarchical Decomposition')
print('═'*72)
display(master_df[MASTER_COLS].round(4))

# ── Determine overall best
master_ranked = master_df.sort_values(
    ['Test Macro Recall', 'Stage 3 Recall', 'Train-Test Gap'],
    ascending=[False, False, True]
).reset_index(drop=True)

BEST_EXP    = master_ranked.iloc[0]['Experiment']
BEST_LABEL  = master_ranked.iloc[0]['Model']
BEST_MODEL  = best_model_smote if 'SMOTE' in BEST_EXP else best_model_hier
BEST_X_TR   = X_train_full
BEST_X_TE   = X_test_full

print(f'    Test Macro Recall : {master_ranked.iloc[0]["Test Macro Recall"]:.4f}')
print(f'    Stage 3 Recall    : {master_ranked.iloc[0]["Stage 3 Recall"]:.4f}')
print(f'    Train-Test Gap    : {master_ranked.iloc[0]["Train-Test Gap"]:.4f}')


In [ ]:
# §14 — Detailed Visualisations
#
# Plots produced (all saved to RESULTS_DIR):
# 1. Confusion matrices: Train & Test for both best SMOTE and Hierarchical
# 2. Performance comparison bar chart (Test metrics)
# 3. Stage-wise recall comparison heatmap
# 4. Train vs Test macro recall comparison (all 6 SMOTE models)
# 5. Class distribution: original vs SMOTE-augmented (1 fold)
# 6. SHAP feature importance (if best model is tree-based)

y_pred_smote_tr = best_model_smote.predict(X_train_full)
y_pred_smote_te = best_model_smote.predict(X_test_full)
y_pred_hier_tr  = best_model_hier.predict(X_train_full)
y_pred_hier_te  = best_model_hier.predict(X_test_full)

display_names = [f'S{s}\n{STAGE_NAMES[s][:8]}' for s in STAGE_LABELS]

# ─── PLOT 1: Side-by-side confusion matrices (SMOTE vs Hierarchical)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
combos = [
    (y_train, y_pred_smote_tr, f'SMOTE ({best_name_smote}) — Train',    axes[0,0]),
    (y_test,  y_pred_smote_te, f'SMOTE ({best_name_smote}) — Test',     axes[0,1]),
    (y_train, y_pred_hier_tr,  'Hierarchical S1 — Train',               axes[1,0]),
    (y_test,  y_pred_hier_te,  'Hierarchical S1 — Test',                axes[1,1]),
]
for y_true, y_pred, title, ax in combos:
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=display_names,
        cmap='Blues', ax=ax, colorbar=False, normalize='true',
        values_format='.2f',
    )
    ax.set_title(title, fontsize=10, fontweight='bold')

plt.suptitle('Confusion Matrices — SMOTE vs Hierarchical Decomposition',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '03_confusion_matrices.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved: confusion matrices')

# ─── PLOT 2: Performance Comparison Bar Chart
metrics_compare = {
    'Macro Recall':     ['Test Macro Recall',    'Test Macro Recall'],
    'Macro Precision':  ['Test Macro Precision',  'Test Macro Precision'],
    'Macro F1':         ['Test Macro F1',          'Test Macro F1'],
    'Balanced Acc':     ['Test Balanced Acc',      'Test Balanced Acc'],
}
smote_vals = [master_df[master_df['Experiment'] == 'Exp A — SMOTE'].iloc[0][c]
              for c in ['Test Macro Recall', 'Test Macro Precision', 'Test Macro F1', 'Test Balanced Acc']]
hier_vals  = [master_df[master_df['Experiment'] == 'Exp B — Hierarchical S1'].iloc[0][c]
              for c in ['Test Macro Recall', 'Test Macro Precision', 'Test Macro F1', 'Test Balanced Acc']]

x = np.arange(4)
w = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - w/2, smote_vals, w, label=f'SMOTE ({best_name_smote})',
               color='#1565C0', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + w/2, hier_vals,  w, label='Hierarchical S1',
               color='#E53935', alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(['Macro\nRecall', 'Macro\nPrecision', 'Macro\nF1', 'Balanced\nAcc'], fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Experiment Comparison — Test Set Performance', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='Random baseline')
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_experiment_comparison_bar.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Plot 2 saved: performance bar chart')

# ─── PLOT 3: Stage-wise Recall Heatmap
stage_recall_data = {
    f'SMOTE\n({best_name_smote})': [
        recall_score((y_test == s).astype(int),
                     (y_pred_smote_te == s).astype(int), zero_division=0)
        for s in STAGE_LABELS],
    'Hierarchical\nS1': [
        recall_score((y_test == s).astype(int),
                     (y_pred_hier_te == s).astype(int), zero_division=0)
        for s in STAGE_LABELS],
}
df_heat = pd.DataFrame(stage_recall_data,
    index=[f'Stage {s}: {STAGE_NAMES[s]}' for s in STAGE_LABELS])

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, vmin=0, vmax=1,
            annot_kws={'fontsize': 12})
ax.set_title('Stage-wise Recall — SMOTE vs Hierarchical (Test Set)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '05_stagewise_recall_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3 saved: stage-wise recall heatmap')

# ─── PLOT 4: All SMOTE Models — Train vs Test Macro Recall
fig, ax = plt.subplots(figsize=(10, 5))
models_s  = results_smote['Model'].tolist()
train_s   = results_smote['Train Macro Recall'].tolist()
test_s    = results_smote['Test Macro Recall'].tolist()
x4 = np.arange(len(models_s))
ax.bar(x4 - 0.2, train_s, 0.4, label='Train', color='#1976D2', alpha=0.8)
ax.bar(x4 + 0.2, test_s,  0.4, label='Test',  color='#FF7043', alpha=0.8)
ax.set_xticks(x4)
ax.set_xticklabels(models_s, rotation=20, ha='right')
ax.set_ylabel('Macro Recall')
ax.set_ylim(0, 1.1)
ax.set_title('SMOTE Experiment — All Models: Train vs Test Macro Recall',
             fontsize=12, fontweight='bold')
ax.legend()
# Mark best
best_idx = models_s.index(best_name_smote)
ax.bar(best_idx + 0.2, test_s[best_idx], 0.4, color='#FFD700',
       alpha=1.0, label='Best', zorder=5)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '06_smote_all_models.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Plot 4 saved: all SMOTE models comparison')

# ─── PLOT 5: Class Distribution Before vs After SMOTE
from imblearn.over_sampling import SMOTE as _SMOTE
from sklearn.preprocessing import RobustScaler as _RS

# Quick demo on training data (for visualisation only)
_X_demo = X_train_full[[c for c in SELECTED_FEATURES if c != MAIN_TARGET]].fillna(
    X_train_full[[c for c in SELECTED_FEATURES if c != MAIN_TARGET]].median())
_X_demo = _X_demo.select_dtypes(include=[np.number])
_X_s    = _RS().fit_transform(_X_demo)
_sm     = _SMOTE(sampling_strategy='not majority', random_state=RANDOM_STATE, k_neighbors=5)
try:
    _, _y_aug = _sm.fit_resample(_X_s, y_train)
    fig, axes5 = plt.subplots(1, 2, figsize=(12, 4))
    for ax5, (y_plot, title) in zip(axes5, [
        (y_train, 'Before SMOTE (Training Set)'),
        (pd.Series(_y_aug), 'After SMOTE'),
    ]):
        counts = y_plot.value_counts().sort_index()
        colors5 = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
        bars5 = ax5.bar([f'S{s}\n{STAGE_NAMES[s][:8]}' for s in STAGE_LABELS],
                        [counts.get(s, 0) for s in STAGE_LABELS], color=colors5, alpha=0.85)
        ax5.set_title(title, fontsize=11, fontweight='bold')
        ax5.set_ylabel('Count')
        for b5 in bars5:
            ax5.text(b5.get_x() + b5.get_width()/2, b5.get_height() + 1,
                     str(int(b5.get_height())), ha='center', va='bottom', fontsize=9)
    plt.suptitle('Class Distribution: SMOTE Effect', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, '07_smote_class_distribution.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot 5 saved: class distribution')
except Exception as e:
    print(f'SMOTE demo failed: {e}')

# ─── PLOT 6: Feature Importance (tree-based models)
feat_names_plot = [c for c in SELECTED_FEATURES if c not in [FIRM_COL, YEAR_COL]]

def get_feature_importance(model, feat_names):
    """Extract feature importances from pipeline's clf step."""
    try:
        clf = model.named_steps.get('clf')
        if clf is None:
            return None
        if hasattr(clf, 'feature_importances_'):
            imp = pd.Series(clf.feature_importances_, index=feat_names[:len(clf.feature_importances_)])
            return imp.sort_values(ascending=False)
    except Exception:
        return None
    return None

imp = get_feature_importance(best_model_smote, feat_names_plot)
if imp is not None:
    fig, ax = plt.subplots(figsize=(10, max(5, len(imp)*0.35)))
    imp.head(20).sort_values().plot(kind='barh', ax=ax, color='#1565C0', alpha=0.85)
    ax.set_title(f'Feature Importance — {best_name_smote} (SMOTE)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, '08_feature_importance.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot 6 saved: feature importance')
else:
    print('Feature importance not available for this model type.')



## §15-XAI — Twelve-Module Explainable AI (XAI) Audit

The winning model (selected by Test Macro Recall) is subjected to a twelve-module explainability audit across two dimensions: **scope** (global vs local) and **purpose** (attribution, validation, error analysis).

| Module | Technique | Scope | Purpose |
|--------|-----------|-------|---------|
| XAI-1 | SHAP Bar Summary (all 4 stages) | Global | Mean feature importance per stage |
| XAI-2 | SHAP Beeswarm (all 4 stages) | Global | Feature value direction and magnitude |
| XAI-3 | SHAP Waterfall (per-firm) | Local | Individual prediction explanation |
| XAI-4 | SHAP Force Plot (interactive) | Local | Stakeholder-facing visual |
| XAI-5 | SHAP Dependence Plots | Global | Non-linear and interaction effects |
| XAI-6 | LIME (model-agnostic) | Local | Independent second-opinion attribution |
| XAI-7 | Partial Dependence Plots | Global | Marginal effect of each ratio on P(stage) |
| XAI-8 | Global Surrogate Decision Tree | Global | IF-THEN rule extraction |
| XAI-9 | SHAP for Hierarchical Layer 1 | Global | Low- vs high-severity split drivers |
| XAI-10 | Cross-Method Summary Heatmap | Global | SHAP, LIME, surrogate agreement |
| XAI-11 | Bootstrap Stability Analysis | Validation | Attribution robustness across subsamples |
| XAI-12 | Misclassification SHAP Deep-Dive | Error analysis | Attribution differences for Stage 3 errors |


In [ ]:
# §15-XAI — Setup: Install LIME & build shared preprocessing helper
#
# We need:
#   • lime            — model-agnostic local explanations
#   • shap (already in §1)
#
# Shared helper: _xai_preprocess(model, X_df)
#   → transforms a DataFrame through every pipeline step EXCEPT smote + clf
#   → returns numpy array safe for SHAP / LIME / PDP
#
# _xai_feat_names : feature names after DropIdColumns (firm / year removed)

!pip install lime -q

import shap, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score
warnings.filterwarnings('ignore')

# ── Identify SMOTE best model (XAI always anchored to SMOTE best for
#    TreeExplainer compatibility; Hierarchical layer gets its own XAI-9)
_xai_model      = best_model_smote
_xai_model_name = best_name_smote

# ── Build preprocessing-only pipeline (drop smote + clf steps)
_skip_steps = {'smote', 'clf'}
_preproc_steps = [(n, s) for n, s in _xai_model.steps if n not in _skip_steps]
_preproc_pipe  = Pipeline(_preproc_steps)

# ── Feature names after DropIdColumns transformer
_xai_feat_names = [c for c in SELECTED_FEATURES if c not in [FIRM_COL, YEAR_COL]]

# ── Transform train & test sets
X_train_t = _preproc_pipe.transform(X_train_full)
X_test_t  = _preproc_pipe.transform(X_test_full)

# ── Build SHAP TreeExplainer on the clf step
_clf_step  = _xai_model.named_steps['clf']
_explainer = shap.TreeExplainer(_clf_step)

# ── Compute SHAP values on test set
_shap_raw = _explainer.shap_values(X_test_t)

# Normalise to dict {stage: (N, F) array}
def _parse_shap(raw):
    if isinstance(raw, list):
        return {s: raw[s] for s in range(4)}
    elif hasattr(raw, 'ndim') and raw.ndim == 3:
        return {s: raw[:, :, s] for s in range(4)}
    else:   # binary fallback
        return {1: raw}

_sv = _parse_shap(_shap_raw)

# ── Compute SHAP Explanation object (new API) for waterfall / force
_shap_exp = _explainer(X_test_t)   # shape: (N, F, C)

# ── Locate test-set examples per stage (correctly predicted preferred)
_stage_idx = {}
for s in STAGE_LABELS:
    correct = np.where((y_test.values == s) & (y_pred_smote_te == s))[0]
    any_s   = np.where(y_test.values == s)[0]
    _stage_idx[s] = int(correct[0]) if len(correct) else (int(any_s[0]) if len(any_s) else None)

print(f'  Model       : {_xai_model_name}')
print(f'  Features    : {len(_xai_feat_names)}')
print(f'  Test shape  : {X_test_t.shape}')
print(f'  SHAP stages : {list(_sv.keys())}')
for s, idx in _stage_idx.items():
    label = '✓ correct' if idx is not None and y_pred_smote_te[idx] == s else '~ any'
    print(f'  Stage {s} example index: {idx}  {label}')


In [ ]:
# §15-XAI-1 — SHAP Bar Summary: Global Feature Importance for All 4 Stages
#
# Why: The original §15 only shows Stage 1 and Stage 3.
#      Stage 0 (Healthy) and Stage 2 (Mild Liquidity) are equally important
#      to understand the full distress spectrum.
# Each bar = mean |SHAP value| across all test-set firms for that class.

stage_colors = {
    0: '#2196F3',   # blue   — Healthy
    1: '#4CAF50',   # green  — Profit Reduction
    2: '#FF9800',   # orange — Mild Liquidity
    3: '#F44336',   # red    — Severe Liquidity
}

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flat

for ax, stage in zip(axes, STAGE_LABELS):
    sv = _sv[stage]
    mean_abs = np.abs(sv).mean(axis=0)
    feat_imp = pd.Series(mean_abs, index=_xai_feat_names).nlargest(12).sort_values()

    feat_imp.plot(kind='barh', ax=ax,
                  color=stage_colors[stage], alpha=0.85, edgecolor='white')
    ax.set_title(f'Stage {stage}: {STAGE_NAMES[stage]}',
                 fontsize=12, fontweight='bold', color=stage_colors[stage])
    ax.set_xlabel('Mean |SHAP Value|', fontsize=10)
    ax.tick_params(axis='y', labelsize=9)
    ax.axvline(0, color='gray', linewidth=0.5)
    for bar in ax.patches:
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.4f}', va='center', fontsize=7.5)

plt.suptitle(f'SHAP Global Feature Importance — All 4 Distress Stages\n'
             f'Model: {_xai_model_name} (SMOTE)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'xai_01_shap_bar_all_stages.png'),
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# §15-XAI-2 — SHAP Beeswarm: Direction + Magnitude for All 4 Stages
#
# Why: Beeswarm shows not just *which* features matter, but *how* their
#      values drive predictions (red dot = high feature value, blue = low).
#      Critical for financial interpretation (e.g. high Current Ratio
#      reduces Stage-3 SHAP → protective factor).

fig, axes = plt.subplots(2, 2, figsize=(18, 16))
axes = axes.flat

for ax, stage in zip(axes, STAGE_LABELS):
    plt.sca(ax)
    shap.summary_plot(
        _sv[stage], X_test_t,
        feature_names=_xai_feat_names,
        plot_type='dot',
        max_display=10,
        show=False,
        color_bar=True,
    )
    ax.set_title(f'Stage {stage}: {STAGE_NAMES[stage]}',
                 fontsize=11, fontweight='bold')
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle(f'SHAP Beeswarm — All 4 Stages (Red=High Feature Value, Blue=Low)\n'
             f'Model: {_xai_model_name}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'xai_02_shap_beeswarm_all_stages.png'),
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# §15-XAI-3 — SHAP Waterfall: Local Explanation for One Firm per Stage
#
# Why: Waterfall plots answer "WHY was THIS specific firm classified as
#      Stage X?" — the most important question a regulator or analyst asks.
# Each bar = one feature's contribution; E[f(x)] = base value (average
# model output); f(x) = final model output for this firm.
#
# We pick one representative firm per stage (correctly predicted if available).

fig, axes = plt.subplots(2, 2, figsize=(18, 16))
axes = axes.flat

for ax, stage in zip(axes, STAGE_LABELS):
    idx = _stage_idx[stage]
    if idx is None:
        ax.text(0.5, 0.5, f'No Stage {stage} samples in test set',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Stage {stage}: {STAGE_NAMES[stage]}', fontsize=11)
        continue

    plt.sca(ax)
    # _shap_exp has shape (N, F, C) — select stage class
    shap.plots.waterfall(
        _shap_exp[idx, :, stage],
        max_display=12,
        show=False,
    )
    correct_tag = '✓ Correct' if y_pred_smote_te[idx] == stage else '✗ Misclassified'
    ax.set_title(
        f'Stage {stage}: {STAGE_NAMES[stage]}\n'
        f'Firm #{idx} | True={stage}, Pred={y_pred_smote_te[idx]} | {correct_tag}',
        fontsize=10, fontweight='bold'
    )

plt.suptitle(f'SHAP Waterfall — Individual Firm Explanations (One per Stage)\n'
             f'Model: {_xai_model_name}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'xai_03_shap_waterfall_per_stage.png'),
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# §15-XAI-4 — SHAP Force Plot: Interactive HTML for Stage-3 Firm
#
# Why: Force plots are the most intuitive SHAP visualisation for non-technical
#      stakeholders. Red arrows = features pushing toward Stage 3;
#      Blue arrows = features pulling away. Saved as interactive HTML.

shap.initjs()

idx_s3 = _stage_idx[3]
if idx_s3 is not None:
    # Per-class expected value
    ev_s3 = _explainer.expected_value
    if isinstance(ev_s3, (list, np.ndarray)):
        ev_s3 = ev_s3[3]

    force_html = shap.force_plot(
        ev_s3,
        _sv[3][idx_s3],
        X_test_t[idx_s3],
        feature_names=_xai_feat_names,
        show=False,
        matplotlib=False,
    )
    out_html = os.path.join(RESULTS_DIR, 'xai_04_force_plot_stage3.html')
    shap.save_html(out_html, force_html)
    print('  Open the HTML file in a browser to view the interactive plot.')

    # Static matplotlib version for report embedding
    fig, ax = plt.subplots(figsize=(16, 3))
    shap.force_plot(
        ev_s3,
        _sv[3][idx_s3],
        X_test_t[idx_s3],
        feature_names=_xai_feat_names,
        show=False,
        matplotlib=True,
    )
    plt.title(f'SHAP Force Plot — Stage 3 Firm #{idx_s3} ({_xai_model_name})',
              fontsize=11, fontweight='bold', pad=20)
    plt.savefig(os.path.join(RESULTS_DIR, 'xai_04_force_plot_stage3_static.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No Stage-3 test sample found — skipping force plot.')


In [ ]:
# §15-XAI-5 — SHAP Dependence Plots: Feature × Stage Interactions
#
# Why: Dependence plots reveal non-linear relationships between a feature
#      value and its SHAP contribution, and automatically highlight the
#      strongest interacting second feature (coloured dots).
#
# We pick the top-2 features per stage (by mean |SHAP|) → 2×4 = 8 subplots.

fig, axes = plt.subplots(4, 2, figsize=(16, 22))

for row, stage in enumerate(STAGE_LABELS):
    sv = _sv[stage]
    mean_abs = np.abs(sv).mean(axis=0)
    top2_idx = np.argsort(mean_abs)[::-1][:2]

    for col, feat_idx in enumerate(top2_idx):
        ax = axes[row, col]
        feat_name = _xai_feat_names[feat_idx]
        shap.dependence_plot(
            feat_idx, sv, X_test_t,
            feature_names=_xai_feat_names,
            ax=ax, show=False,
            dot_size=20, alpha=0.7,
        )
        ax.set_title(
            f'Stage {stage}: {STAGE_NAMES[stage]}\n'
            f'Feature: {feat_name}',
            fontsize=9, fontweight='bold'
        )
        ax.tick_params(labelsize=8)

plt.suptitle(f'SHAP Dependence Plots — Top-2 Features per Stage\nModel: {_xai_model_name}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'xai_05_shap_dependence_all_stages.png'),
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# §15-XAI-6 — LIME: Model-Agnostic Local Explanations
#
# Why: LIME builds a local linear surrogate around a single prediction.
#      It is model-agnostic (works even if TreeExplainer fails on a
#      pipeline) and provides a second independent view of feature importance.
#      SHAP + LIME agreement on the same features = high confidence.
#
# We explain one correctly-predicted Stage-3 and one Stage-0 (Healthy) firm.

from lime.lime_tabular import LimeTabularExplainer

# ── Wrap pipeline predict_proba for preprocessed numpy input
def _lime_predict(X_np):
    """LIME passes raw numpy arrays; we reconstruct a DataFrame."""
    df_tmp = pd.DataFrame(X_np, columns=_xai_feat_names)
    return _xai_model.predict_proba(
        pd.concat([df_tmp,
                   pd.DataFrame(np.zeros((len(df_tmp), 2)),
                                columns=[FIRM_COL, YEAR_COL])], axis=1)
        [SELECTED_FEATURES]
    )

# Simpler wrapper: pass already-transformed data directly to clf
def _lime_predict_t(X_np):
    """Already-transformed data straight to clf."""
    return _clf_step.predict_proba(X_np)

lime_explainer = LimeTabularExplainer(
    training_data=X_train_t,
    feature_names=_xai_feat_names,
    class_names=[STAGE_NAMES[s] for s in STAGE_LABELS],
    mode='classification',
    discretize_continuous=True,
    random_state=RANDOM_STATE,
)

for stage, label_tag in [(3, 'Stage3_SevereLiquidity'), (0, 'Stage0_Healthy')]:
    idx = _stage_idx[stage]
    if idx is None:
        print(f'No {STAGE_NAMES[stage]} sample — skipping LIME.')
        continue

    lime_exp = lime_explainer.explain_instance(
        X_test_t[idx],
        _lime_predict_t,
        num_features=12,
        top_labels=4,
        num_samples=1000,
    )

    # ── Matplotlib figure for the target stage
    fig = lime_exp.as_pyplot_figure(label=stage)
    correct_tag = '✓ Correct' if y_pred_smote_te[idx] == stage else '✗ Misclassified'
    fig.suptitle(
        f'LIME — {STAGE_NAMES[stage]}\n'
        f'Firm #{idx} | True={stage}, Pred={y_pred_smote_te[idx]} | {correct_tag}\n'
        f'Model: {_xai_model_name}',
        fontsize=11, fontweight='bold'
    )
    fig.tight_layout()
    fname = f'xai_06_lime_{label_tag}.png'
    fig.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Save interactive HTML
    lime_exp.save_to_file(
        os.path.join(RESULTS_DIR, f'xai_06_lime_{label_tag}.html')
    )



In [ ]:
# §15-XAI-7 — Partial Dependence Plots (PDP): Population-Level Feature Effects
#
# Why: PDP shows the average predicted probability of a class as one feature
#      varies (marginalising over all other features). Unlike SHAP dependence
#      plots (which show the SHAP value), PDP shows the raw probability —
#      more intuitive for clinical/financial stakeholders.
#
# We show PDP for Stage 3 and Stage 0 using the top-3 SHAP features each.

from sklearn.inspection import PartialDependenceDisplay

X_test_df = pd.DataFrame(X_test_t, columns=_xai_feat_names)

for stage, color in [(3, '#F44336'), (0, '#2196F3')]:
    top3_names = (
        pd.Series(np.abs(_sv[stage]).mean(axis=0), index=_xai_feat_names)
        .nlargest(3)
        .index.tolist()
    )

    fig, axes_pdp = plt.subplots(1, 3, figsize=(18, 5))
    try:
        disp = PartialDependenceDisplay.from_estimator(
            _clf_step,
            X_test_df,
            features=top3_names,
            feature_names=_xai_feat_names,
            target=stage,
            kind='average',
            ax=axes_pdp,
            line_kw={'color': color, 'linewidth': 2},
            pd_line_kw={'color': color},
        )
        for ax_p, feat in zip(axes_pdp, top3_names):
            ax_p.set_title(feat, fontsize=10, fontweight='bold')
            ax_p.set_ylabel(f'P(Stage {stage})', fontsize=9)
            ax_p.tick_params(labelsize=8)

        plt.suptitle(
            f'Partial Dependence Plots — Stage {stage}: {STAGE_NAMES[stage]}\n'
            f'Top-3 SHAP Features | Model: {_xai_model_name}',
            fontsize=12, fontweight='bold'
        )
        plt.tight_layout()
        fname = f'xai_07_pdp_stage{stage}.png'
        plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f'PDP Stage {stage} failed: {e}')


In [ ]:
# §15-XAI-8 — Global Surrogate Decision Tree
#
# Why: A shallow DT trained to MIMIC the best model's predictions yields a
#      fully transparent, rule-based approximation. Fidelity measures how
#      well the surrogate copies the black box (not ground truth accuracy).
#      Examiners and regulators value this: "show me the rules your model
#      learned" → this answers that question interpretably.
#
# Depth 4 → at most 15 leaf rules; readable in a report.

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

surrogate = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=5,
    random_state=RANDOM_STATE
)
# Fit on BLACK-BOX PREDICTIONS (not ground truth)
surrogate.fit(X_test_t, y_pred_smote_te)

fidelity  = accuracy_score(y_pred_smote_te, surrogate.predict(X_test_t))
ground_acc = accuracy_score(y_test, surrogate.predict(X_test_t))
print(f'Surrogate fidelity  (mimics black box)  : {fidelity:.4f}')
print(f'Surrogate accuracy  (vs ground truth)   : {ground_acc:.4f}')

# ── Tree plot
fig, ax = plt.subplots(figsize=(26, 11))
plot_tree(
    surrogate,
    feature_names=_xai_feat_names,
    class_names=[STAGE_NAMES[s] for s in STAGE_LABELS],
    filled=True, rounded=True,
    fontsize=8,
    ax=ax,
    impurity=True,
    proportion=False,
    precision=3,
)
plt.title(
    f'Global Surrogate Decision Tree — Approximates {_xai_model_name}\n'
    f'Fidelity = {fidelity:.4f}  |  Depth = 4  |  Interpretable Rules',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'xai_08_surrogate_tree.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# ── Print extracted IF-THEN rules (first 4000 chars)
rules_text = export_text(surrogate, feature_names=_xai_feat_names)
print('\n── Surrogate Decision Rules (excerpt) ─────────────────────────────')
print(rules_text[:4000])

# Save full rules to text file
rules_path = os.path.join(RESULTS_DIR, 'xai_08_surrogate_rules.txt')
with open(rules_path, 'w') as f:
    f.write(f'Global Surrogate Decision Tree Rules\n')
    f.write(f'Mimics: {_xai_model_name} (SMOTE)\n')
    f.write(f'Fidelity: {fidelity:.4f}\n\n')
    f.write(rules_text)


In [ ]:
# §15-XAI-9 — SHAP for Hierarchical Model: Layer 1 ({0,1} vs {2,3})
#
# Why: The original §15 skips SHAP entirely if the Hierarchical model wins.
#      This cell always runs SHAP on Hierarchical Layer 1 regardless of which
#      model is the overall best. Layer 1 makes the most impactful binary
#      decision: low-severity vs high-severity — understanding it is crucial.
#
# Note: Layer-specific feature sets differ from SMOTE's full feature set.
#       We use the Layer-1 feature set defined in §8b.

try:
    # ── Extract the Layer-1 classifier from the hierarchical pipeline
    #    (variable name depends on your §12 implementation)
    if 'best_model_hier' in dir() and best_model_hier is not None:
        # Try common attribute names used in §12
        _l1_clf = None
        for attr in ['l1_clf', 'layer1_model', 'clf_l1']:
            _l1_clf = getattr(best_model_hier, attr, None)
            if _l1_clf is not None:
                break

        # Fallback: inspect steps if it's a Pipeline
        if _l1_clf is None and hasattr(best_model_hier, 'steps'):
            for name, step in best_model_hier.steps:
                if hasattr(step, 'feature_importances_'):
                    _l1_clf = step
                    break

        if _l1_clf is None:
            raise ValueError('Cannot locate Layer-1 classifier in best_model_hier. '
                             'Ensure best_model_hier has a .l1_clf or .layer1_model attribute.')

        # ── Extract the final classifier from the _l1_clf pipeline for TreeExplainer
        _l1_final_clf = _l1_clf.named_steps['clf']

        # ── Layer-1 feature set (from §8b: FEATS_L1)
        # FEATS_L1 already excludes FIRM_COL and YEAR_COL.
        _l1_feat_names = FEATS_L1

        # ── Build preprocessing pipeline for Layer-1 from _l1_clf's steps
        # Extract all steps from the _l1_clf pipeline except the final classifier.
        _l1_preproc_pipe = Pipeline(_l1_clf.steps[:-1])

        # ── Transform test set using the Layer-1 specific preprocessing pipeline
        # Pass the full X_test_full which contains ID_COLS required by some preprocessors
        X_test_l1_t = _l1_preproc_pipe.transform(X_test_full)

        # ── Build Layer-1 SHAP explainer using the extracted classifier
        _l1_explainer = shap.TreeExplainer(_l1_final_clf)
        _l1_shap = _l1_explainer.shap_values(X_test_l1_t)

        # Layer 1 is binary: class 1 = High-Severity ({Stage 2, Stage 3})
        if isinstance(_l1_shap, list):
            _l1_sv = _l1_shap[1]
        elif hasattr(_l1_shap, 'ndim') and _l1_shap.ndim == 3:
            _l1_sv = _l1_shap[:, :, 1]
        else:
            _l1_sv = _l1_shap

        fig, axes_l1 = plt.subplots(1, 2, figsize=(18, 7))

        # Bar chart
        plt.sca(axes_l1[0])
        shap.summary_plot(_l1_sv, X_test_l1_t,
                          feature_names=_l1_feat_names,
                          plot_type='bar', show=False, max_display=12)
        axes_l1[0].set_title('Layer 1: Feature Importance (Bar)', fontsize=11, fontweight='bold')

        # Beeswarm
        plt.sca(axes_l1[1])
        shap.summary_plot(_l1_sv, X_test_l1_t,
                          feature_names=_l1_feat_names,
                          plot_type='dot', show=False, max_display=12)
        axes_l1[1].set_title('Layer 1: SHAP Direction (Beeswarm)', fontsize=11, fontweight='bold')

        plt.suptitle(
            f'Hierarchical Layer 1 SHAP — Low Severity {{0,1}} vs High Severity {{2,3}}\n'
            f'Model: {best_name_hier}',
            fontsize=12, fontweight='bold'
        )
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, 'xai_09_shap_hier_layer1.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()

    else:
        print('best_model_hier not found — Hierarchical experiment may not have run.')

except Exception as e:
    print(f'XAI-9 (Hierarchical SHAP) failed: {e}')
    print('This is non-critical. SMOTE SHAP (XAI-1 to XAI-5) remains valid.')
    import traceback; traceback.print_exc()

In [ ]:
# §15-XAI-10 — XAI Summary: Top Features Across Stages & Methods
#
# Produces a consolidated DataFrame showing:
#   • SHAP rank (1 = most important) per stage
#   • LIME rank for Stage 3 (from last LIME run)
#   • Surrogate tree feature usage (split count)
# This table is the deliverable for the FYP report XAI discussion section.

TOP_N = 10

# ── SHAP rank per stage
shap_ranks = {}
for stage in STAGE_LABELS:
    mean_abs = np.abs(_sv[stage]).mean(axis=0)
    ranked   = pd.Series(mean_abs, index=_xai_feat_names).rank(ascending=False).astype(int)
    shap_ranks[f'SHAP Rank Stage {stage}\n({STAGE_NAMES[stage][:10]})'] = ranked

shap_rank_df = pd.DataFrame(shap_ranks)

# ── Surrogate feature usage (number of splits per feature)
from sklearn.tree import _tree
def _count_splits(tree, feat_names):
    tree_ = tree.tree_
    counts = {f: 0 for f in feat_names}
    for node in range(tree_.node_count):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            counts[feat_names[tree_.feature[node]]] += 1
    return pd.Series(counts)

surrogate_splits = _count_splits(surrogate, _xai_feat_names)
shap_rank_df['Surrogate\nSplit Count'] = surrogate_splits

# ── Overall importance score: average SHAP rank across stages (lower = more important)
shap_rank_df['Avg SHAP\nRank (↓ better)'] = (
    shap_rank_df[[c for c in shap_rank_df.columns if 'SHAP Rank' in c]]
    .mean(axis=1)
    .round(1)
)

# ── Show top-10 by average SHAP rank
top_feats = shap_rank_df.nsmallest(TOP_N, 'Avg SHAP\nRank (↓ better)')

print('\n' + '═'*72)
print('  XAI SUMMARY — Top Features Across All Stages & Methods')
print('═'*72)
display(top_feats.round(2))

# Save to CSV
top_feats.to_csv(os.path.join(RESULTS_DIR, 'xai_10_feature_importance_summary.csv'))

# ── Final heatmap of SHAP ranks
fig, ax = plt.subplots(figsize=(12, max(6, TOP_N * 0.55)))
heat_data = top_feats[[c for c in top_feats.columns if 'SHAP Rank' in c]]
sns.heatmap(
    heat_data, annot=True, fmt='d', cmap='RdYlGn_r',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Rank (1 = most important)'},
    annot_kws={'fontsize': 10},
)
ax.set_title(
    f'SHAP Feature Rank Heatmap — Top {TOP_N} Features across All Stages\n'
    f'Model: {_xai_model_name}  |  Rank 1 = Most Important',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('')
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=9, rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'xai_10_shap_rank_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f'  Outputs saved to: {RESULTS_DIR}')


In [ ]:
#  §15-XAI-11 — STABILITY ANALYSIS


# §15-XAI-11 — Explanation Stability via Bootstrap
# Purpose: Validate SHAP explanations are robust across data subsets
# Method: Bootstrap sampling → measure coefficient of variation (CV)
# Output: Features with CV < 0.3 have stable, trustworthy explanations

print("XAI VALIDATION — EXPLANATION STABILITY ANALYSIS")

try:

    # Configuration
    n_bootstrap = 10
    sample_size = min(200, len(X_test_t))

    print(f"   Bootstrap iterations: {n_bootstrap}")
    print(f"   Sample size per iteration: {sample_size}")

    # Store SHAP importances across bootstraps
    bootstrap_shap = {stage: [] for stage in range(4)}

    np.random.seed(42)
    for boot_iter in range(n_bootstrap):
        # Sample with replacement
        boot_idx = np.random.choice(len(X_test_t), sample_size, replace=True)
        X_boot = X_test_t[boot_idx]

        # Compute SHAP values
        shap_boot = _explainer.shap_values(X_boot)

        # Calculate mean absolute SHAP per feature per stage
        for stage in range(4):
            if isinstance(shap_boot, list):
                mean_abs = np.abs(shap_boot[stage]).mean(axis=0)
            else:
                mean_abs = np.abs(shap_boot[:, :, stage]).mean(axis=0)
            bootstrap_shap[stage].append(mean_abs)


    # Calculate stability metrics (coefficient of variation)

    stability_results = []
    for stage in range(4):
        # importance_matrix: (n_bootstrap, n_features)
        importance_matrix = np.array(bootstrap_shap[stage])
        mean_imp = importance_matrix.mean(axis=0)
        std_imp = importance_matrix.std(axis=0)
        cv = std_imp / (mean_imp + 1e-10)  # Coefficient of variation

        # Get stable features (lowest CV)
        stable_idx = np.argsort(cv)[:5]

        stability_results.append({
            'stage': stage,
            'stage_name': STAGE_NAMES[stage],
            'mean_cv': cv.mean(),
            'median_cv': np.median(cv),
            'stable_features': [(_xai_feat_names[i], mean_imp[i], cv[i]) for i in stable_idx]
        })

    # Visualize stability
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    axes = axes.flatten()

    for stage in range(4):
        ax = axes[stage]
        importance_matrix = np.array(bootstrap_shap[stage])

        # Get top 10 features by mean importance
        top_10_idx = np.argsort(importance_matrix.mean(axis=0))[-10:][::-1]

        # Prepare boxplot data
        box_data = [importance_matrix[:, i] for i in top_10_idx]
        box_labels = [(_xai_feat_names[i]) for i in top_10_idx]

        # Create boxplot
        bp = ax.boxplot(box_data, labels=box_labels, vert=False, patch_artist=True)

        # Color boxes by stability (green = stable, red = unstable)
        for i, (patch, idx) in enumerate(zip(bp['boxes'], top_10_idx)):
            cv_val = (importance_matrix[:, idx].std() /
                     (importance_matrix[:, idx].mean() + 1e-10))
            color = 'lightgreen' if cv_val < 0.3 else 'lightyellow' if cv_val < 0.5 else 'lightcoral'
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax.set_xlabel('SHAP Importance (10 Bootstrap Samples)', fontsize=11, fontweight='bold')
        ax.set_title(f'Stage {stage}: {STAGE_NAMES[stage]}\nExplanation Stability (Green=Stable)',
                    fontsize=12, fontweight='bold')
        ax.tick_params(axis='y', labelsize=9)
        ax.grid(alpha=0.3, axis='x')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'xai_11_stability_analysis.png'),
               dpi=200, bbox_inches='tight')
    plt.show()

    # Save stability metrics
    stability_df = pd.DataFrame([
        {
            'Stage': r['stage'],
            'Stage_Name': r['stage_name'],
            'Mean_CV': r['mean_cv'],
            'Median_CV': r['median_cv']
        }
        for r in stability_results
    ])
    stability_df.to_csv(os.path.join(RESULTS_DIR, 'xai_stability_metrics.csv'), index=False)

    # Print stability report

    for result in stability_results:
        print(f"   Overall Stability: Mean CV = {result['mean_cv']:.3f}")

        for feat, imp, cv in result['stable_features']:
            reliability = "✓ Highly Reliable" if cv < 0.3 else "○ Reliable" if cv < 0.5 else "⚠ Use Caution"
            print(f"      {feat:<30} CV={cv:.3f} ({reliability})")

    print("\n" + "=" * 80)

except Exception as e:
    import traceback
    traceback.print_exc()

In [ ]:
# §15-XAI-12 — MISCLASSIFICATION ANALYSIS

# §15-XAI-12 — Misclassification Deep-Dive: Why Does the Model Fail?
# Purpose: Identify systematic error patterns by comparing correct vs incorrect predictions
# Method: SHAP pattern comparison between correctly and incorrectly classified samples
# Output: Features that differ most → reveal why model makes mistakes

print("XAI ERROR ANALYSIS — MISCLASSIFICATION PATTERNS")

try:

    # Get predictions
    y_pred_full = _xai_model.predict(X_test_full)
    correct_mask = (y_pred_full == y_test)

    # Overall statistics
    n_total = len(y_test)
    n_correct = correct_mask.sum()
    n_wrong = (~correct_mask).sum()

    print(f"   Total samples: {n_total}")

    # Analyze by stage
    print("\n" + "=" * 80)
    print("CONFUSION BREAKDOWN BY STAGE")

    misclass_patterns = []

    for true_stage in range(4):
        stage_mask = (y_test == true_stage)
        n_stage = stage_mask.sum()

        if n_stage == 0:
            continue

        stage_correct = correct_mask[stage_mask].sum()
        stage_wrong = (~correct_mask[stage_mask]).sum()


        if stage_wrong > 0:
            # Where are they misclassified?
            wrong_indices = np.where(stage_mask & ~correct_mask)[0]
            wrong_preds = y_pred_full[wrong_indices]

            print(f"\n   Error Distribution:")
            for pred_stage in range(4):
                if pred_stage == true_stage:
                    continue
                count = (wrong_preds == pred_stage).sum()
                if count > 0:
                    pct = count / stage_wrong * 100
                    print(f"      Stage {true_stage} → Stage {pred_stage}: {count} cases ({pct:.1f}% of errors)")

                    misclass_patterns.append({
                        'true_stage': true_stage,
                        'pred_stage': pred_stage,
                        'count': count,
                        'pct_of_stage_errors': pct
                    })

    # Deep-dive: Stage 3 (most critical)
    print("\n" + "=" * 80)
    print("DEEP DIVE: STAGE 3 (SEVERE LIQUIDITY DISTRESS)")

    stage3_mask = (y_test == 3)
    stage3_correct_mask = stage3_mask & correct_mask
    stage3_wrong_mask = stage3_mask & ~correct_mask

    n_s3_correct = stage3_correct_mask.sum()
    n_s3_wrong = stage3_wrong_mask.sum()

    if n_s3_correct > 0 and n_s3_wrong > 0:

        # Get SHAP values for Stage 3
        # _sv is a dictionary, as defined in XAI setup (uDKm-ETga5Gm)
        # It's consistently {stage: array} for all stages.
        shap_s3 = _sv[3]

        # Compare SHAP patterns
        shap_correct_mean = shap_s3[stage3_correct_mask].mean(axis=0)
        shap_wrong_mean = shap_s3[stage3_wrong_mask].mean(axis=0)
        shap_diff = np.abs(shap_correct_mean - shap_wrong_mean)

        # Top differentiating features
        top_diff_idx = np.argsort(shap_diff)[-10:][::-1]

        # Visualization
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

        # Left: Pattern Comparison
        features = [_xai_feat_names[i] for i in top_diff_idx]
        correct_vals = [shap_correct_mean[i] for i in top_diff_idx]
        wrong_vals = [shap_wrong_mean[i] for i in top_diff_idx]

        x = np.arange(len(features))
        width = 0.35

        ax1.barh(x - width/2, correct_vals, width,
                label=f'Correct Predictions (n={n_s3_correct})',
                color='green', alpha=0.8)
        ax1.barh(x + width/2, wrong_vals, width,
                label=f'Misclassified (n={n_s3_wrong})',
                color='red', alpha=0.8)

        ax1.set_yticks(x)
        ax1.set_yticklabels(features, fontsize=10)
        ax1.set_xlabel('Mean SHAP Value', fontsize=12, fontweight='bold')
        ax1.set_title('Stage 3: Feature Patterns\nCorrect vs Misclassified',
                     fontsize=13, fontweight='bold')
        ax1.legend(loc='best', fontsize=10)
        ax1.axvline(x=0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)
        ax1.invert_yaxis()
        ax1.grid(axis='x', alpha=0.3)

        # Right: Difference Magnitude
        diff_vals = [shap_diff[i] for i in top_diff_idx]
        colors = ['darkred' if d > np.median(shap_diff[top_diff_idx]) else 'orange'
                 for d in diff_vals]

        ax2.barh(features, diff_vals, alpha=0.8, color=colors)
        ax2.set_xlabel('|SHAP Difference|', fontsize=12, fontweight='bold')
        ax2.set_title('Features Most Different\nBetween Correct & Misclassified',
                     fontsize=13, fontweight='bold')
        ax2.invert_yaxis()
        ax2.grid(axis='x', alpha=0.3)

        # Add reference line at median
        median_diff = np.median(diff_vals)
        ax2.axvline(median_diff, color='gray', linestyle='--', alpha=0.5,
                   label=f'Median: {median_diff:.4f}')
        ax2.legend()

        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, 'xai_12_misclassification_analysis_stage3.png'),
                   dpi=200, bbox_inches='tight')
        plt.show()

        # Print actionable insights
        print("\n" + "=" * 80)
        print("\nTop 5 features where correct and incorrect predictions differ most:\n")

        for rank, feat_idx in enumerate(top_diff_idx[:5], 1):
            feat = _xai_feat_names[feat_idx]
            correct_shap = shap_correct_mean[feat_idx]
            wrong_shap = shap_wrong_mean[feat_idx]
            diff = shap_diff[feat_idx]

            # Determine direction of difference
            if abs(correct_shap) > abs(wrong_shap):
                direction = "STRONGER signal in correct predictions"
            else:
                direction = "WEAKER signal in correct predictions"

            print(f"{rank}. {feat}")
            print()

        print("   Misclassified Stage 3 companies show systematically different feature")

        # Save misclassification patterns
        if misclass_patterns:
            misclass_df = pd.DataFrame(misclass_patterns)
            misclass_df.to_csv(os.path.join(RESULTS_DIR, 'xai_misclassification_patterns.csv'),
                              index=False)

    elif n_s3_wrong == 0:
    else:
        print(f"   (Correct: {n_s3_correct}, Wrong: {n_s3_wrong})")


except Exception as e:
    import traceback
    traceback.print_exc()

In [ ]:
# §16 — Save All Results to Drive

# ── Classification reports
print('=== Classification Report: SMOTE Best Model (Test Set) ===')
print(classification_report(
    y_test, y_pred_smote_te,
    target_names=[STAGE_NAMES[s] for s in STAGE_LABELS],
    zero_division=0
))

print('=== Classification Report: Hierarchical S1 (Test Set) ===')
print(classification_report(
    y_test, y_pred_hier_te,
    target_names=[STAGE_NAMES[s] for s in STAGE_LABELS],
    zero_division=0
))

# ── Save master comparison CSV
master_save = master_df[MASTER_COLS].round(4)
master_save.to_csv(os.path.join(RESULTS_DIR, 'master_comparison.csv'), index=False)

# ── Save per-experiment results CSVs
results_smote.to_csv(os.path.join(RESULTS_DIR, 'results_smote.csv'), index=False)
results_hier.to_csv( os.path.join(RESULTS_DIR, 'results_hierarchical.csv'), index=False)

# ── Save feature config

# ── Summary printout
print('\n' + '═'*72)
print('  FINAL SUMMARY — PSX Financial Distress Early Warning System')
print('═'*72)
display(master_df[MASTER_COLS].round(4))

best_row = master_ranked.iloc[0]
print(f'   Test Macro Recall : {best_row["Test Macro Recall"]:.4f}')
print(f'   Test Macro F1     : {best_row["Test Macro F1"]:.4f}')
print(f'   Stage 1 Recall    : {best_row["Stage 1 Recall"]:.4f}')
print(f'   Stage 3 Recall    : {best_row["Stage 3 Recall"]:.4f}')
print(f'   Train-Test Gap    : {best_row["Train-Test Gap"]:.4f}')

print("""
Methodology Summary
────────────────────────────────────────────────────────────────
Data       : PSX non-financial firms | Fiscal years (post T+1 shift)
Target     : Main_Distress_T1 — 4-class next-year label (Stage 0-3)
Targets    : T+1 created via firm-wise shift(-1) on sorted panel data
Split      : Last 20% per class (chronologically) → test set
             Preserves temporal ordering; guarantees class coverage in test
Imputation : Firm-wise rolling mean (window=5, past obs only)
             → global median fallback for residual NaN
Outliers   : Winsorise at 1st/99th pct (fitted on train only)
Scaling    : RobustScaler (IQR-based, outlier resistant)
Features   : MI Top-6 per binary T+1 flag (1vsRest) → union
             Main_Distress (current year) always added
             Missingness filter >30%, Correlation filter ≥0.98
Experiments:
  A (SMOTE)         : Flat 4-class + data augmentation in pipeline
  B (Hierarchical)  : {0,1} vs {2,3} → Stage 0/1 | Stage 2/3
                      Layer-specific features per sub-problem
                      Exposure-bias fix in sub-model training
Models     : 3 tree (RF, ET, DT) + 3 ensemble (LGBM, XGB, Bagging)
CV         : TimeSeriesSplit(n_splits=5) in all RandomizedSearchCV
N_ITER     : 10 (RandomizedSearchCV iterations)
Scoring    : recall_macro (primary) — early warning requires high recall
"""
)
